# Preparing Eron.csv data for Fine-Tuning




In [62]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [63]:
import pandas as pd

df_enron = pd.read_csv('/content/drive/MyDrive/PhishDetectAI/datasets/raw_data/Enron.csv')

In [64]:
print(df_enron)

                                          subject  \
0                       hpl nom for may 25 , 2001   
1                re : nom / actual vols for 24 th   
2      enron actuals for march 30 - april 1 , 201   
3                       hpl nom for may 30 , 2001   
4                       hpl nom for june 1 , 2001   
...                                           ...   
29762                          confidence is back   
29763                       important information   
29764                           vys - make itnger   
29765        the best thing come in large parcels   
29766                           it works amaziing   

                                                    body  label  
0      ( see attached file : hplno 525 . xls )\r\n- h...      0  
1      - - - - - - - - - - - - - - - - - - - - - - fo...      0  
2      estimated actuals\r\nmarch 30 , 2001\r\nno flo...      0  
3      ( see attached file : hplno 530 . xls )\r\n- h...      0  
4      ( see attached file : hpln

# Certain part of exploration

In [65]:
print(df_enron.shape)
print(df_enron.columns)
df_enron.head()

(29767, 3)
Index(['subject', 'body', 'label'], dtype='object')


,subject,body,label
0,"hpl nom for may 25 , 2001",( see attached file : hplno 525 . xls )\r\n- h...,0
1,re : nom / actual vols for 24 th,- - - - - - - - - - - - - - - - - - - - - - fo...,0
2,"enron actuals for march 30 - april 1 , 201","estimated actuals\r\nmarch 30 , 2001\r\nno flo...",0
3,"hpl nom for may 30 , 2001",( see attached file : hplno 530 . xls )\r\n- h...,0
4,"hpl nom for june 1 , 2001",( see attached file : hplno 601 . xls )\r\n- h...,0


In [66]:
df_enron.isnull().sum()


,0
subject,198
body,0
label,0


In [67]:
df_enron.duplicated().any()

np.False_

In [68]:
print(df_enron["label"].value_counts())

label
0    15791
1    13976
Name: count, dtype: int64


In [69]:
df_enron["subject_length"] = df_enron["subject"].str.len()
df_enron["body_length"] = df_enron["body"].str.len()

print(df_enron[["subject_length", "body_length"]].describe())

       subject_length    body_length
count    29569.000000   29767.000000
mean        34.301532    1487.822421
std         47.831388    4202.647954
min          1.000000       1.000000
25%         19.000000     332.000000
50%         30.000000     706.000000
75%         44.000000    1565.000000
max       7170.000000  230120.000000


In [70]:
print(df_enron.dtypes)

subject            object
body               object
label               int64
subject_length    float64
body_length         int64
dtype: object


# Cleaning the Data

In [71]:
#Filling null value with string

df_enron["subject"] = df_enron["subject"].fillna("")

In [72]:
df_enron.isnull().sum()

,0
subject,0
body,0
label,0
subject_length,198
body_length,0


In [73]:
print("Empty subjects:", (df_enron["subject"].str.strip() == "").sum())
print("Empty bodies:", (df_enron["body"].str.strip() == "").sum())

Empty subjects: 198
Empty bodies: 0


In [74]:
empty_emails = (
    (df_enron["subject"].str.strip() == "") &
    (df_enron["body"].str.strip() == "")
).sum()

print("Completely empty emails:", empty_emails)

Completely empty emails: 0


In [75]:
df_enron = df_enron.drop(columns=["subject_length", "body_length"])

In [76]:
print(df_enron.dtypes)

subject    object
body       object
label       int64
dtype: object


In [77]:
print(df_enron["label"].unique())
print(df_enron["label"].value_counts())

[0 1]
label
0    15791
1    13976
Name: count, dtype: int64


In [78]:
def converted_example(example):

    if example["label"] == 0:
        label = "legitimate"
    else:
        label = "phishing"

    return {
        "instruction": "Classify the following email as phishing or legitimate.",
        "input": f"Subject: {example['subject']}\n\nBody: {example['body']}",
        "output": label
    }

In [79]:
# Hugging Face Dataset
from datasets import Dataset

dataset = Dataset.from_pandas(df_enron)

In [80]:
dataset = dataset.map(converted_example)

Map:   0%|          | 0/29767 [00:00<?, ? examples/s]

In [81]:
print(dataset[0])

{'subject': 'hpl nom for may 25 , 2001', 'body': '( see attached file : hplno 525 . xls )\r\n- hplno 525 . xls', 'label': 0, 'instruction': 'Classify the following email as phishing or legitimate.', 'input': 'Subject: hpl nom for may 25 , 2001\n\nBody: ( see attached file : hplno 525 . xls )\r\n- hplno 525 . xls', 'output': 'legitimate'}


In [82]:
dataset = dataset.map(
    converted_example,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/29767 [00:00<?, ? examples/s]

In [83]:
print(dataset)

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 29767
})


In [84]:
dataset = dataset.train_test_split(
    test_size=0.2,
    seed=42
)

In [85]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 23813
    })
    test: Dataset({
        features: ['instruction', 'input', 'output'],
        num_rows: 5954
    })
})


In [86]:
test = dataset["train"][0]

print(type(test))
print(test)

<class 'dict'>
{'instruction': 'Classify the following email as phishing or legitimate.', 'input': "Subject: barrister thambo\n\nBody: i am barrister andrew thambo an attorney of clifford alexander ' s attorney in johannesburg , south africa . a special client of mine mandated me to look for over sea ' s company , agent or an individual who can act as his over sea ' s partner ' s in his proposed investment in real estate / property in their country and also investment in some blue chip company ' s\r\nhowever the companies or individual ' s involvement will be properly paid for although the term ' s and condition ' s are negotiable ?\r\nexpecting your swift response . our second chamber ' s email is andy _ thambo _ jhb @ yahoo . com\r\nregard ' s\r\nbarrister andrew thambo", 'output': 'phishing'}


In [87]:
display(dataset['train'].to_pandas().head(10))

,instruction,input,output
0,Classify the following email as phishing or le...,Subject: barrister thambo\n\nBody: i am barris...,phishing
1,Classify the following email as phishing or le...,Subject: nomination 6 / 1 / 2000 - eastrans\n\...,legitimate
2,Classify the following email as phishing or le...,Subject: re : wip report for week 09 / 10 / 01...,legitimate
3,Classify the following email as phishing or le...,Subject: start date : 12 / 16 / 01 ; hourahead...,legitimate
4,Classify the following email as phishing or le...,Subject: fw : re : cash vs . lc\n\nBody: - - -...,legitimate
5,Classify the following email as phishing or le...,Subject: clal 1 s : put some zip back in your ...,phishing
6,Classify the following email as phishing or le...,Subject: california update 3 / 27 / 01\n\nBody...,legitimate
7,Classify the following email as phishing or le...,Subject: miningnews . net newsletter - monday ...,phishing
8,Classify the following email as phishing or le...,Subject: schedule for next week\n\nBody: to al...,legitimate
9,Classify the following email as phishing or le...,Subject: reminder\n\nBody: might body more roo...,phishing


In [88]:
# Saving Dataset in JSONL

import json
import os

with open("/content/drive/MyDrive/PhishDetectAI/datasets/prepared_data/train.jsonl", "w", encoding="utf-8") as f:
    for example in dataset["train"]:
        f.write(
            json.dumps(example, ensure_ascii=False) + "\n"
        )

print("Train dataset saved.")

Train dataset saved.


In [89]:
# Validate
import json

with open("/content/drive/MyDrive/PhishDetectAI/datasets/prepared_data/train.jsonl", "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        try:
            example = json.loads(line)
            print("Line", line_number, "OK")
        except json.JSONDecodeError:
            print("Line", line_number, "ERROR")

Streaming output truncated to the last 5000 lines.
Line 18814 OK
Line 18815 OK
Line 18816 OK
Line 18817 OK
Line 18818 OK
Line 18819 OK
Line 18820 OK
Line 18821 OK
Line 18822 OK
Line 18823 OK
Line 18824 OK
Line 18825 OK
Line 18826 OK
Line 18827 OK
Line 18828 OK
Line 18829 OK
Line 18830 OK
Line 18831 OK
Line 18832 OK
Line 18833 OK
Line 18834 OK
Line 18835 OK
Line 18836 OK
Line 18837 OK
Line 18838 OK
Line 18839 OK
Line 18840 OK
Line 18841 OK
Line 18842 OK
Line 18843 OK
Line 18844 OK
Line 18845 OK
Line 18846 OK
Line 18847 OK
Line 18848 OK
Line 18849 OK
Line 18850 OK
Line 18851 OK
Line 18852 OK
Line 18853 OK
Line 18854 OK
Line 18855 OK
Line 18856 OK
Line 18857 OK
Line 18858 OK
Line 18859 OK
Line 18860 OK
Line 18861 OK
Line 18862 OK
Line 18863 OK
Line 18864 OK
Line 18865 OK
Line 18866 OK
Line 18867 OK
Line 18868 OK
Line 18869 OK
Line 18870 OK
Line 18871 OK
Line 18872 OK
Line 18873 OK
Line 18874 OK
Line 18875 OK
Line 18876 OK
Line 18877 OK
Line 18878 OK
Line 18879 OK
Line 18880 OK
Line 18881 

In [90]:
import json

with open(
    "/content/drive/MyDrive/PhishDetectAI/datasets/prepared_data/test.jsonl",
    "w",
    encoding="utf-8"
) as f:
    for example in dataset["test"]:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Test dataset saved.")

Test dataset saved.


In [91]:
import json

with open(
    "/content/drive/MyDrive/PhishDetectAI/datasets/prepared_data/test.jsonl",
    "r",
    encoding="utf-8"
) as f:
    for line_number, line in enumerate(f, start=1):
        try:
            example = json.loads(line)
            print("Line", line_number, "OK")
        except json.JSONDecodeError:
            print("Line", line_number, "ERROR")

Streaming output truncated to the last 5000 lines.
Line 955 OK
Line 956 OK
Line 957 OK
Line 958 OK
Line 959 OK
Line 960 OK
Line 961 OK
Line 962 OK
Line 963 OK
Line 964 OK
Line 965 OK
Line 966 OK
Line 967 OK
Line 968 OK
Line 969 OK
Line 970 OK
Line 971 OK
Line 972 OK
Line 973 OK
Line 974 OK
Line 975 OK
Line 976 OK
Line 977 OK
Line 978 OK
Line 979 OK
Line 980 OK
Line 981 OK
Line 982 OK
Line 983 OK
Line 984 OK
Line 985 OK
Line 986 OK
Line 987 OK
Line 988 OK
Line 989 OK
Line 990 OK
Line 991 OK
Line 992 OK
Line 993 OK
Line 994 OK
Line 995 OK
Line 996 OK
Line 997 OK
Line 998 OK
Line 999 OK
Line 1000 OK
Line 1001 OK
Line 1002 OK
Line 1003 OK
Line 1004 OK
Line 1005 OK
Line 1006 OK
Line 1007 OK
Line 1008 OK
Line 1009 OK
Line 1010 OK
Line 1011 OK
Line 1012 OK
Line 1013 OK
Line 1014 OK
Line 1015 OK
Line 1016 OK
Line 1017 OK
Line 1018 OK
Line 1019 OK
Line 1020 OK
Line 1021 OK
Line 1022 OK
Line 1023 OK
Line 1024 OK
Line 1025 OK
Line 1026 OK
Line 1027 OK
Line 1028 OK
Line 1029 OK
Line 1030 OK
Line 1

In [92]:
import json
from collections import Counter

TRAIN_PATH = "/content/drive/MyDrive/PhishDetectAI/datasets/prepared_data/train.jsonl"

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f]

print("Total training samples:", len(train_data))

print("\nLabel distribution:")
print(Counter(item["output"] for item in train_data))

print("\n--- Example labeled legitimate ---")
for item in train_data:
    if item["output"] == "legitimate":
        print(item["input"][:500])
        print("Output:", item["output"])
        break

print("\n--- Example labeled phishing ---")
for item in train_data:
    if item["output"] == "phishing":
        print(item["input"][:500])
        print("Output:", item["output"])
        break

Total training samples: 23813

Label distribution:
Counter({'legitimate': 12640, 'phishing': 11173})

--- Example labeled legitimate ---
Subject: nomination 6 / 1 / 2000 - eastrans

Body: revisions for 6 / 1 / 2000 on behalf of bruce mcmills :
this is to nominate 33 , 450 mmbtu / d into eastrans effective 6 / 1 / 2000
redeliveries will be made as follows :
25 , 000 into pg & e at carthage
8 , 450 from fuels cotton valley
duke residue sales
6 / 1 4 , 520 mmbtu
6 / 2 5 , 690 mmbtu
6 / 3 5 , 105 mmbtu
- - - - - - - - - - - - - - - - - - - - - - forwarded by william e . speckels / gcs / cec / pec on
05 / 26 / 2000 01 : 38 p
Output: legitimate

--- Example labeled phishing ---
Subject: barrister thambo

Body: i am barrister andrew thambo an attorney of clifford alexander ' s attorney in johannesburg , south africa . a special client of mine mandated me to look for over sea ' s company , agent or an individual who can act as his over sea ' s partner ' s in his proposed investment in real est